# 09 — Supplier & Procurement Analysis

## Sales Order, Backlog & Inventory Analytics

### Objective

This notebook evaluates supplier and procurement performance using supplier master data and purchase-order transactions.

The analysis focuses on:

- Overall procurement KPIs
- Purchase order status
- Procurement value and quantity
- Supplier performance
- Supplier reliability
- Supplier quality ratings
- Contract vs actual lead-time performance
- Supplier delivery delays
- On-time receipt performance
- Open purchase orders
- Preferred vs non-preferred suppliers
- Procurement trends
- Supplier concentration
- High-risk suppliers
- Procurement risk areas

### Data Quality Rule

Open purchase orders may have missing:

- `actual_receipt_date`
- `supplier_delay_days`

These are **structural missing values**, because the purchase orders have not yet been received.

They will not be treated as data-quality errors and will be excluded from completed-delivery performance calculations where appropriate.

In [1]:
# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("Pandas version :", pd.__version__)
print("NumPy version  :", np.__version__)

print(
    "\nSupplier & Procurement Analysis environment initialized successfully."
)

Pandas version : 3.0.6
NumPy version  : 2.5.3

Supplier & Procurement Analysis environment initialized successfully.


## Step 2 — Define Feature-Engineered Data Path

Define the project directory and the location of the feature-engineered datasets.

Supplier & Procurement Analysis will use the analysis-ready datasets produced by `04_Feature_Engineering.ipynb`.

In [2]:
# Define project paths

project_path = Path.cwd().parent

featured_data_path = (
    project_path / "data" / "featured"
)

print("Project Path       :", project_path)
print("Featured Data Path :", featured_data_path)

print(
    "\nFeatured Data Folder Exists :",
    featured_data_path.exists()
)

Project Path       : c:\Users\Admin\OneDrive\Documents\Sales_Order_Backlog_Inventory_Analytics
Featured Data Path : c:\Users\Admin\OneDrive\Documents\Sales_Order_Backlog_Inventory_Analytics\data\featured

Featured Data Folder Exists : True


## Step 3 — Load Supplier & Procurement Analysis Datasets

Load the feature-engineered datasets required for supplier and procurement analysis.

The analysis uses:

- Purchase Orders
- Suppliers
- Products
- Warehouses

These datasets provide purchase-order transactions, supplier attributes, product information, and warehouse details required for procurement-performance analysis.

In [3]:
# Load Supplier & Procurement Analysis datasets

purchase_orders = pd.read_csv(
    featured_data_path / "purchase_orders_featured.csv",
    parse_dates=[
        "po_date",
        "expected_date",
        "actual_receipt_date"
    ]
)

suppliers = pd.read_csv(
    featured_data_path / "suppliers_featured.csv"
)

products = pd.read_csv(
    featured_data_path / "products_featured.csv"
)

warehouses = pd.read_csv(
    featured_data_path / "warehouses_featured.csv"
)


procurement_datasets = {
    "Purchase Orders": purchase_orders,
    "Suppliers": suppliers,
    "Products": products,
    "Warehouses": warehouses
}


print(
    "Supplier & Procurement Analysis datasets loaded successfully.\n"
)

for name, df in procurement_datasets.items():

    print(
        f"{name:<20} "
        f"{df.shape[0]:>7,} rows × "
        f"{df.shape[1]:>2} columns"
    )

Supplier & Procurement Analysis datasets loaded successfully.

Purchase Orders        8,500 rows × 23 columns
Suppliers                100 rows ×  7 columns
Products                 500 rows × 10 columns
Warehouses                 6 rows ×  5 columns


## Step 4 — Validate Procurement Analysis Fields

Validate the Purchase Orders dataset before beginning procurement-performance analysis.

The validation confirms the availability of:

- Purchase order identifiers
- Supplier and product keys
- Warehouse information
- Ordered and received quantities
- Procurement value
- Contract lead time
- Expected and actual receipt dates
- Supplier delay
- Purchase order status

Structural missing values for open purchase orders are retained and will be handled appropriately during delivery-performance analysis.

In [4]:
# Validate procurement analysis fields

required_po_columns = [
    "po_id",
    "product_id",
    "warehouse_id",
    "supplier_id",
    "po_date",
    "ordered_qty",
    "received_qty",
    "unit_cost",
    "po_value",
    "contract_lead_time_days",
    "expected_date",
    "actual_receipt_date",
    "supplier_delay_days",
    "po_status"
]


missing_columns = [
    col
    for col in required_po_columns
    if col not in purchase_orders.columns
]


print("PROCUREMENT DATA VALIDATION\n")

print(
    f"Total Purchase Orders   : "
    f"{len(purchase_orders):,}"
)

print(
    f"Total Columns           : "
    f"{purchase_orders.shape[1]}"
)

print(
    f"\nRequired Columns Missing: "
    f"{len(missing_columns)}"
)


if missing_columns:
    print("Missing Columns :", missing_columns)
else:
    print(
        "All required procurement analysis fields are available."
    )


print("\nPurchase Order Status Distribution:")

print(
    purchase_orders[
        "po_status"
    ].value_counts(dropna=False)
)


print("\nStructural Missing Values:")

print(
    purchase_orders[
        [
            "actual_receipt_date",
            "supplier_delay_days"
        ]
    ]
    .isna()
    .sum()
)

PROCUREMENT DATA VALIDATION

Total Purchase Orders   : 8,500
Total Columns           : 23

Required Columns Missing: 0
All required procurement analysis fields are available.

Purchase Order Status Distribution:
po_status
Partially Received    8470
Open                    30
Name: count, dtype: int64

Structural Missing Values:
actual_receipt_date    30
supplier_delay_days    30
dtype: int64


## Step 5 — Overall Procurement KPIs

Calculate the core procurement KPIs to establish the overall purchasing and receipt-performance baseline.

The KPIs include:

- Total Purchase Orders
- Total Procurement Value
- Total Units Ordered
- Total Units Received
- Outstanding Units
- Quantity Receipt Rate
- Average Purchase Order Value
- Unique Suppliers
- Unique Products Procured
- Open Purchase Orders
- Partially Received Purchase Orders

In [5]:
# Overall Procurement KPIs

total_purchase_orders = len(purchase_orders)

total_procurement_value = purchase_orders[
    "po_value"
].sum()

total_units_ordered = purchase_orders[
    "ordered_qty"
].sum()

total_units_received = purchase_orders[
    "received_qty"
].sum()

outstanding_units = (
    total_units_ordered
    - total_units_received
)

quantity_receipt_rate = (
    total_units_received
    / total_units_ordered
    * 100
)

average_po_value = (
    total_procurement_value
    / total_purchase_orders
)

unique_suppliers = purchase_orders[
    "supplier_id"
].nunique()

unique_products_procured = purchase_orders[
    "product_id"
].nunique()

open_purchase_orders = (
    purchase_orders["po_status"]
    .eq("Open")
    .sum()
)

partially_received_pos = (
    purchase_orders["po_status"]
    .eq("Partially Received")
    .sum()
)


print("OVERALL PROCUREMENT KPIs\n")

print(
    f"Total Purchase Orders        : "
    f"{total_purchase_orders:,}"
)

print(
    f"Total Procurement Value      : "
    f"{total_procurement_value:,.2f}"
)

print(
    f"Total Units Ordered          : "
    f"{total_units_ordered:,}"
)

print(
    f"Total Units Received         : "
    f"{total_units_received:,}"
)

print(
    f"Outstanding Units            : "
    f"{outstanding_units:,}"
)

print(
    f"Quantity Receipt Rate        : "
    f"{quantity_receipt_rate:.2f}%"
)

print(
    f"Average Purchase Order Value : "
    f"{average_po_value:,.2f}"
)

print(
    f"Unique Suppliers             : "
    f"{unique_suppliers:,}"
)

print(
    f"Unique Products Procured     : "
    f"{unique_products_procured:,}"
)

print(
    f"Open Purchase Orders         : "
    f"{open_purchase_orders:,}"
)

print(
    f"Partially Received POs       : "
    f"{partially_received_pos:,}"
)

OVERALL PROCUREMENT KPIs

Total Purchase Orders        : 8,500
Total Procurement Value      : 6,715,682,357.79
Total Units Ordered          : 2,012,081
Total Units Received         : 1,879,255
Outstanding Units            : 132,826
Quantity Receipt Rate        : 93.40%
Average Purchase Order Value : 790,080.28
Unique Suppliers             : 100
Unique Products Procured     : 500
Open Purchase Orders         : 30
Partially Received POs       : 8,470


## Step 6 — Purchase Order Status Analysis

Analyze purchase orders by procurement status to understand the current state of supplier receipts.

For each PO status, evaluate:

- Number of purchase orders
- Ordered quantity
- Received quantity
- Outstanding quantity
- Procurement value
- Share of total procurement value
- Quantity receipt rate

This provides visibility into partially received and still-open procurement commitments.

In [6]:
# Purchase Order Status Analysis

po_status_summary = (
    purchase_orders
    .groupby("po_status")
    .agg(
        po_count=("po_id", "count"),
        ordered_qty=("ordered_qty", "sum"),
        received_qty=("received_qty", "sum"),
        procurement_value=("po_value", "sum")
    )
    .reset_index()
)


po_status_summary["outstanding_qty"] = (
    po_status_summary["ordered_qty"]
    - po_status_summary["received_qty"]
)


po_status_summary["receipt_rate_pct"] = (
    po_status_summary["received_qty"]
    / po_status_summary["ordered_qty"]
    * 100
)


po_status_summary["procurement_value_share_pct"] = (
    po_status_summary["procurement_value"]
    / total_procurement_value
    * 100
)


po_status_summary = po_status_summary[
    [
        "po_status",
        "po_count",
        "ordered_qty",
        "received_qty",
        "outstanding_qty",
        "procurement_value",
        "procurement_value_share_pct",
        "receipt_rate_pct"
    ]
]


po_status_summary = (
    po_status_summary
    .sort_values(
        "procurement_value",
        ascending=False
    )
    .reset_index(drop=True)
)


print("PURCHASE ORDER STATUS ANALYSIS\n")

print(
    po_status_summary.to_string(
        index=False,
        formatters={
            "procurement_value":
                lambda x: f"{x:,.2f}",

            "procurement_value_share_pct":
                lambda x: f"{x:.2f}%",

            "receipt_rate_pct":
                lambda x: f"{x:.2f}%"
        }
    )
)

PURCHASE ORDER STATUS ANALYSIS

         po_status  po_count  ordered_qty  received_qty  outstanding_qty procurement_value procurement_value_share_pct receipt_rate_pct
Partially Received      8470      2005208       1879255           125953  6,693,927,716.35                      99.68%           93.72%
              Open        30         6873             0             6873     21,754,641.44                       0.32%            0.00%


## Step 7 — Supplier Delivery Performance

Evaluate supplier delivery performance using purchase orders with completed receipt information.

Open purchase orders are excluded because their actual receipt dates and supplier delay values are not yet available.

The analysis measures:

- Purchase orders evaluated
- On-time / early receipts
- Delayed receipts
- On-time receipt rate
- Supplier delay rate
- Average supplier delay
- Median supplier delay
- Maximum supplier delay

This establishes the overall supplier delivery reliability baseline.

In [7]:
# Supplier Delivery Performance

received_pos = purchase_orders[
    purchase_orders["supplier_delay_days"].notna()
].copy()


evaluated_pos = len(received_pos)

on_time_supplier_pos = (
    received_pos["supplier_delay_days"] <= 0
).sum()

delayed_supplier_pos = (
    received_pos["supplier_delay_days"] > 0
).sum()


supplier_on_time_rate = (
    on_time_supplier_pos
    / evaluated_pos
    * 100
)

supplier_delay_rate = (
    delayed_supplier_pos
    / evaluated_pos
    * 100
)


avg_supplier_delay = (
    received_pos["supplier_delay_days"].mean()
)

median_supplier_delay = (
    received_pos["supplier_delay_days"].median()
)

max_supplier_delay = (
    received_pos["supplier_delay_days"].max()
)


print("SUPPLIER DELIVERY PERFORMANCE\n")

print(
    f"Purchase Orders Evaluated : "
    f"{evaluated_pos:,}"
)

print(
    f"On-Time / Early Receipts  : "
    f"{on_time_supplier_pos:,}"
)

print(
    f"Delayed Receipts          : "
    f"{delayed_supplier_pos:,}"
)

print(
    f"On-Time Receipt Rate      : "
    f"{supplier_on_time_rate:.2f}%"
)

print(
    f"Supplier Delay Rate       : "
    f"{supplier_delay_rate:.2f}%"
)

print(
    f"Average Supplier Delay    : "
    f"{avg_supplier_delay:.2f} days"
)

print(
    f"Median Supplier Delay     : "
    f"{median_supplier_delay:.2f} days"
)

print(
    f"Maximum Supplier Delay    : "
    f"{max_supplier_delay:.0f} days"
)

SUPPLIER DELIVERY PERFORMANCE

Purchase Orders Evaluated : 8,470
On-Time / Early Receipts  : 5,535
Delayed Receipts          : 2,935
On-Time Receipt Rate      : 65.35%
Supplier Delay Rate       : 34.65%
Average Supplier Delay    : 0.84 days
Median Supplier Delay     : 0.00 days
Maximum Supplier Delay    : 17 days


## Step 8 — Supplier Delay Severity Analysis

Analyze delayed purchase orders by the severity of supplier delay.

Delayed receipts are classified into:

- 1–2 Days Late
- 3–5 Days Late
- 6–10 Days Late
- 11+ Days Late

For each severity level, evaluate:

- Number of delayed purchase orders
- Ordered quantity
- Received quantity
- Procurement value
- Average delay
- Maximum delay
- Share of delayed purchase orders

This helps distinguish minor supplier delays from more significant procurement disruptions.

In [8]:
# Supplier Delay Severity Analysis

delayed_pos = received_pos[
    received_pos["supplier_delay_days"] > 0
].copy()


delayed_pos["delay_severity"] = pd.cut(
    delayed_pos["supplier_delay_days"],
    bins=[0, 2, 5, 10, np.inf],
    labels=[
        "1-2 Days Late",
        "3-5 Days Late",
        "6-10 Days Late",
        "11+ Days Late"
    ],
    include_lowest=True
)


delay_severity_summary = (
    delayed_pos
    .groupby(
        "delay_severity",
        observed=True
    )
    .agg(
        po_count=("po_id", "count"),
        ordered_qty=("ordered_qty", "sum"),
        received_qty=("received_qty", "sum"),
        procurement_value=("po_value", "sum"),
        avg_delay_days=("supplier_delay_days", "mean"),
        max_delay_days=("supplier_delay_days", "max")
    )
    .reset_index()
)


delay_severity_summary["delayed_po_share_pct"] = (
    delay_severity_summary["po_count"]
    / delayed_supplier_pos
    * 100
)


print("SUPPLIER DELAY SEVERITY ANALYSIS\n")

print(
    delay_severity_summary.to_string(
        index=False,
        formatters={
            "procurement_value":
                lambda x: f"{x:,.2f}",

            "avg_delay_days":
                lambda x: f"{x:.2f}",

            "max_delay_days":
                lambda x: f"{x:.0f}",

            "delayed_po_share_pct":
                lambda x: f"{x:.2f}%"
        }
    )
)


print(
    f"\nTotal Delayed Purchase Orders : "
    f"{delay_severity_summary['po_count'].sum():,}"
)

SUPPLIER DELAY SEVERITY ANALYSIS

delay_severity  po_count  ordered_qty  received_qty procurement_value avg_delay_days max_delay_days delayed_po_share_pct
 1-2 Days Late      1872       443837        415867  1,511,613,306.89           1.04              2               63.78%
 3-5 Days Late       227        54900         51545    191,540,697.90           3.93              5                7.73%
6-10 Days Late       322        75674         71012    261,548,205.63           8.04             10               10.97%
 11+ Days Late       514       124454        116554    420,898,179.17          14.21             17               17.51%

Total Delayed Purchase Orders : 2,935


## Step 9 — Supplier-Level Performance Analysis

Evaluate procurement performance at the individual supplier level.

For each supplier, calculate:

- Purchase orders evaluated
- Ordered quantity
- Received quantity
- Procurement value
- Average supplier delay
- Delayed purchase orders
- On-time / early purchase orders
- On-time receipt rate
- Supplier delay rate

Supplier master attributes are also included:

- Supplier name
- Supplier region
- Quality rating
- Preferred supplier status
- Reliability factor

This analysis helps identify high-performing suppliers and suppliers creating procurement-delivery risk.

In [9]:
# Supplier-Level Performance Analysis

supplier_delivery = (
    received_pos
    .groupby("supplier_id")
    .agg(
        po_count=("po_id", "count"),
        ordered_qty=("ordered_qty", "sum"),
        received_qty=("received_qty", "sum"),
        procurement_value=("po_value", "sum"),
        avg_delay_days=("supplier_delay_days", "mean"),
        delayed_pos=(
            "supplier_delay_days",
            lambda x: (x > 0).sum()
        ),
        on_time_pos=(
            "supplier_delay_days",
            lambda x: (x <= 0).sum()
        )
    )
    .reset_index()
)


supplier_delivery["on_time_rate_pct"] = (
    supplier_delivery["on_time_pos"]
    / supplier_delivery["po_count"]
    * 100
)


supplier_delivery["delay_rate_pct"] = (
    supplier_delivery["delayed_pos"]
    / supplier_delivery["po_count"]
    * 100
)


supplier_delivery = supplier_delivery.merge(
    suppliers[
        [
            "supplier_id",
            "supplier_name",
            "supplier_region",
            "quality_rating",
            "preferred_supplier_flag",
            "reliability_factor"
        ]
    ],
    on="supplier_id",
    how="left"
)


supplier_delivery = (
    supplier_delivery
    .sort_values(
        "on_time_rate_pct",
        ascending=False
    )
    .reset_index(drop=True)
)


print("SUPPLIER-LEVEL PERFORMANCE ANALYSIS\n")

print(
    supplier_delivery[
        [
            "supplier_id",
            "supplier_name",
            "po_count",
            "procurement_value",
            "on_time_pos",
            "delayed_pos",
            "on_time_rate_pct",
            "delay_rate_pct",
            "avg_delay_days",
            "quality_rating",
            "preferred_supplier_flag",
            "reliability_factor"
        ]
    ]
    .head(10)
    .to_string(
        index=False,
        formatters={
            "procurement_value":
                lambda x: f"{x:,.2f}",
            "on_time_rate_pct":
                lambda x: f"{x:.2f}%",
            "delay_rate_pct":
                lambda x: f"{x:.2f}%",
            "avg_delay_days":
                lambda x: f"{x:.2f}"
        }
    )
)


print(
    f"\nSuppliers Evaluated : "
    f"{supplier_delivery['supplier_id'].nunique():,}"
)

print(
    "Missing Supplier Matches :",
    supplier_delivery["supplier_name"].isna().sum()
)

SUPPLIER-LEVEL PERFORMANCE ANALYSIS

supplier_id supplier_name  po_count procurement_value  on_time_pos  delayed_pos on_time_rate_pct delay_rate_pct avg_delay_days  quality_rating preferred_supplier_flag  reliability_factor
     SUP025  Supplier 025        47     43,404,409.40           41            6           87.23%         12.77%          -0.83            3.18                       Y               0.909
     SUP002  Supplier 002        89     81,353,826.53           77           12           86.52%         13.48%          -0.54            4.43                       N               0.965
     SUP026  Supplier 026        47     36,022,747.52           39            8           82.98%         17.02%          -0.64            4.04                       N               0.944
     SUP066  Supplier 066        21     12,503,952.78           17            4           80.95%         19.05%          -0.05            3.80                       Y               0.926
     SUP020  Supplier 020   

## Step 10 — Lowest-Performing Suppliers by On-Time Rate

Identify suppliers with the lowest on-time receipt performance.

The analysis focuses on suppliers with sufficient procurement activity and compares:

- Purchase order count
- Procurement value
- On-time receipt rate
- Delay rate
- Average supplier delay
- Quality rating
- Preferred supplier status
- Reliability factor

This helps identify suppliers that may require performance review, corrective action, or closer procurement monitoring.

In [10]:
# Lowest-Performing Suppliers by On-Time Rate

lowest_on_time_suppliers = (
    supplier_delivery
    .sort_values(
        [
            "on_time_rate_pct",
            "procurement_value"
        ],
        ascending=[True, False]
    )
    .head(10)
    .reset_index(drop=True)
)


print("LOWEST-PERFORMING SUPPLIERS BY ON-TIME RATE\n")

print(
    lowest_on_time_suppliers[
        [
            "supplier_id",
            "supplier_name",
            "po_count",
            "procurement_value",
            "on_time_pos",
            "delayed_pos",
            "on_time_rate_pct",
            "delay_rate_pct",
            "avg_delay_days",
            "quality_rating",
            "preferred_supplier_flag",
            "reliability_factor"
        ]
    ]
    .to_string(
        index=False,
        formatters={
            "procurement_value":
                lambda x: f"{x:,.2f}",

            "on_time_rate_pct":
                lambda x: f"{x:.2f}%",

            "delay_rate_pct":
                lambda x: f"{x:.2f}%",

            "avg_delay_days":
                lambda x: f"{x:.2f}"
        }
    )
)

LOWEST-PERFORMING SUPPLIERS BY ON-TIME RATE

supplier_id supplier_name  po_count procurement_value  on_time_pos  delayed_pos on_time_rate_pct delay_rate_pct avg_delay_days  quality_rating preferred_supplier_flag  reliability_factor
     SUP012  Supplier 012        44     33,483,158.98           17           27           38.64%         61.36%           3.66            3.48                       N               0.621
     SUP089  Supplier 089       257    300,925,064.66          103          154           40.08%         59.92%           4.05            4.52                       N               0.570
     SUP015  Supplier 015        43     44,689,906.80           20           23           46.51%         53.49%           2.58            3.44                       N               0.805
     SUP022  Supplier 022        91     68,276,344.71           43           48           47.25%         52.75%           1.60            4.33                       N               0.807
     SUP078  Supplie

## Step 11 — Preferred vs Non-Preferred Supplier Performance

Compare procurement performance between preferred and non-preferred suppliers.

The analysis evaluates:

- Number of suppliers
- Purchase orders
- Procurement value
- Ordered quantity
- Received quantity
- On-time / early receipts
- Delayed receipts
- On-time receipt rate
- Delay rate
- Average supplier delay

This determines whether preferred-supplier designation is associated with stronger delivery performance.

In [11]:
# Preferred vs Non-Preferred Supplier Performance

preferred_analysis = (
    supplier_delivery
    .groupby("preferred_supplier_flag")
    .agg(
        supplier_count=("supplier_id", "nunique"),
        po_count=("po_count", "sum"),
        ordered_qty=("ordered_qty", "sum"),
        received_qty=("received_qty", "sum"),
        procurement_value=("procurement_value", "sum"),
        on_time_pos=("on_time_pos", "sum"),
        delayed_pos=("delayed_pos", "sum")
    )
    .reset_index()
)


preferred_analysis["on_time_rate_pct"] = (
    preferred_analysis["on_time_pos"]
    / preferred_analysis["po_count"]
    * 100
)


preferred_analysis["delay_rate_pct"] = (
    preferred_analysis["delayed_pos"]
    / preferred_analysis["po_count"]
    * 100
)


# Weighted average delay using PO-level data
preferred_delay = (
    received_pos
    .merge(
        suppliers[
            [
                "supplier_id",
                "preferred_supplier_flag"
            ]
        ],
        on="supplier_id",
        how="left"
    )
    .groupby("preferred_supplier_flag")[
        "supplier_delay_days"
    ]
    .mean()
    .rename("avg_delay_days")
    .reset_index()
)


preferred_analysis = preferred_analysis.merge(
    preferred_delay,
    on="preferred_supplier_flag",
    how="left"
)


print("PREFERRED VS NON-PREFERRED SUPPLIER PERFORMANCE\n")

print(
    preferred_analysis.to_string(
        index=False,
        formatters={
            "procurement_value":
                lambda x: f"{x:,.2f}",

            "on_time_rate_pct":
                lambda x: f"{x:.2f}%",

            "delay_rate_pct":
                lambda x: f"{x:.2f}%",

            "avg_delay_days":
                lambda x: f"{x:.2f}"
        }
    )
)

PREFERRED VS NON-PREFERRED SUPPLIER PERFORMANCE

preferred_supplier_flag  supplier_count  po_count  ordered_qty  received_qty procurement_value  on_time_pos  delayed_pos on_time_rate_pct delay_rate_pct avg_delay_days
                      N              60      5240      1240371       1162718  4,233,248,003.14         3367         1873           64.26%         35.74%           1.00
                      Y              40      3230       764837        716537  2,460,679,713.21         2168         1062           67.12%         32.88%           0.57


## Step 12 — Supplier Region Performance

Analyze procurement and delivery performance across supplier regions.

For each supplier region, evaluate:

- Number of suppliers
- Purchase orders
- Procurement value
- Ordered quantity
- Received quantity
- On-time / early receipts
- Delayed receipts
- On-time receipt rate
- Delay rate
- Average supplier delay

This helps identify whether procurement delivery risk is concentrated in particular supplier regions.

In [12]:
# Supplier Region Performance

supplier_region_data = (
    received_pos
    .merge(
        suppliers[
            [
                "supplier_id",
                "supplier_region"
            ]
        ],
        on="supplier_id",
        how="left"
    )
)


supplier_region_data["on_time_flag"] = (
    supplier_region_data["supplier_delay_days"] <= 0
).astype(int)

supplier_region_data["delayed_flag"] = (
    supplier_region_data["supplier_delay_days"] > 0
).astype(int)


supplier_region_performance = (
    supplier_region_data
    .groupby("supplier_region")
    .agg(
        supplier_count=("supplier_id", "nunique"),
        po_count=("po_id", "count"),
        ordered_qty=("ordered_qty", "sum"),
        received_qty=("received_qty", "sum"),
        procurement_value=("po_value", "sum"),
        on_time_pos=("on_time_flag", "sum"),
        delayed_pos=("delayed_flag", "sum"),
        avg_delay_days=("supplier_delay_days", "mean")
    )
    .reset_index()
)


supplier_region_performance["on_time_rate_pct"] = (
    supplier_region_performance["on_time_pos"]
    / supplier_region_performance["po_count"]
    * 100
)


supplier_region_performance["delay_rate_pct"] = (
    supplier_region_performance["delayed_pos"]
    / supplier_region_performance["po_count"]
    * 100
)


supplier_region_performance = (
    supplier_region_performance
    .sort_values(
        "on_time_rate_pct",
        ascending=False
    )
    .reset_index(drop=True)
)


print("SUPPLIER REGION PERFORMANCE\n")

print(
    supplier_region_performance.to_string(
        index=False,
        formatters={
            "procurement_value":
                lambda x: f"{x:,.2f}",

            "on_time_rate_pct":
                lambda x: f"{x:.2f}%",

            "delay_rate_pct":
                lambda x: f"{x:.2f}%",

            "avg_delay_days":
                lambda x: f"{x:.2f}"
        }
    )
)


print(
    "\nMissing Supplier Region Matches :",
    supplier_region_data[
        "supplier_region"
    ].isna().sum()
)

SUPPLIER REGION PERFORMANCE

supplier_region  supplier_count  po_count  ordered_qty  received_qty procurement_value  on_time_pos  delayed_pos avg_delay_days on_time_rate_pct delay_rate_pct
           East              22      2009       473092        442783  1,612,883,503.42         1353          656           0.66           67.35%         32.65%
          South              15      1050       250585        234778    779,701,682.26          696          354           0.71           66.29%         33.71%
        Central              19      1614       381209        356798  1,158,792,099.47         1063          551           0.66           65.86%         34.14%
          North              18      1478       351468        329810  1,207,279,002.90          953          525           0.97           64.48%         35.52%
           West              26      2319       548854        515086  1,935,271,428.30         1470          849           1.09           63.39%         36.61%

Missing Su

## Step 13 — Product Category Procurement Performance

Analyze procurement performance across product categories.

For each category, evaluate:

- Purchase order count
- Ordered quantity
- Received quantity
- Outstanding quantity
- Procurement value
- Share of procurement value
- Delayed purchase orders
- On-time receipt rate
- Average supplier delay

This identifies product categories with high procurement exposure and categories experiencing greater supplier-delivery risk.

In [13]:
# Product Category Procurement Performance

category_procurement = (
    received_pos
    .merge(
        products[
            [
                "product_id",
                "category"
            ]
        ],
        on="product_id",
        how="left"
    )
)


category_procurement["on_time_flag"] = (
    category_procurement["supplier_delay_days"] <= 0
).astype(int)

category_procurement["delayed_flag"] = (
    category_procurement["supplier_delay_days"] > 0
).astype(int)


category_performance = (
    category_procurement
    .groupby("category")
    .agg(
        po_count=("po_id", "count"),
        ordered_qty=("ordered_qty", "sum"),
        received_qty=("received_qty", "sum"),
        procurement_value=("po_value", "sum"),
        delayed_pos=("delayed_flag", "sum"),
        on_time_pos=("on_time_flag", "sum"),
        avg_delay_days=("supplier_delay_days", "mean")
    )
    .reset_index()
)


category_performance["outstanding_qty"] = (
    category_performance["ordered_qty"]
    - category_performance["received_qty"]
)


category_performance["on_time_rate_pct"] = (
    category_performance["on_time_pos"]
    / category_performance["po_count"]
    * 100
)


category_performance["procurement_value_share_pct"] = (
    category_performance["procurement_value"]
    / category_performance["procurement_value"].sum()
    * 100
)


category_performance = (
    category_performance
    .sort_values(
        "procurement_value",
        ascending=False
    )
    .reset_index(drop=True)
)


print("PRODUCT CATEGORY PROCUREMENT PERFORMANCE\n")

print(
    category_performance[
        [
            "category",
            "po_count",
            "ordered_qty",
            "received_qty",
            "outstanding_qty",
            "procurement_value",
            "procurement_value_share_pct",
            "delayed_pos",
            "on_time_rate_pct",
            "avg_delay_days"
        ]
    ]
    .to_string(
        index=False,
        formatters={
            "procurement_value":
                lambda x: f"{x:,.2f}",

            "procurement_value_share_pct":
                lambda x: f"{x:.2f}%",

            "on_time_rate_pct":
                lambda x: f"{x:.2f}%",

            "avg_delay_days":
                lambda x: f"{x:.2f}"
        }
    )
)


print(
    "\nMissing Product Category Matches :",
    category_procurement["category"].isna().sum()
)

PRODUCT CATEGORY PROCUREMENT PERFORMANCE

              category  po_count  ordered_qty  received_qty  outstanding_qty procurement_value procurement_value_share_pct  delayed_pos on_time_rate_pct avg_delay_days
           Electronics      2225       530317        496702            33615  1,877,535,028.58                      28.05%          810           63.60%           1.03
        Home & Kitchen      1614       390535        365946            24589  1,285,240,452.76                      19.20%          535           66.85%           0.66
       Office Supplies      1558       366463        343332            23131  1,246,989,880.58                      18.63%          528           66.11%           0.80
Health & Personal Care      1545       362001        339137            22864  1,177,752,708.97                      17.59%          524           66.08%           0.72
            Automotive      1528       355892        334138            21754  1,106,409,645.46                      16

## Step 14 — Warehouse Procurement Performance

Analyze procurement and supplier-delivery performance across warehouses.

For each warehouse, evaluate:

- Purchase order count
- Ordered quantity
- Received quantity
- Outstanding quantity
- Procurement value
- Delayed purchase orders
- On-time / early purchase orders
- On-time receipt rate
- Delay rate
- Average supplier delay

This identifies warehouses with greater procurement exposure and inbound supplier-delivery risk.

In [14]:
# Warehouse Procurement Performance

warehouse_procurement = (
    received_pos
    .merge(
        warehouses[
            [
                "warehouse_id",
                "warehouse_name"
            ]
        ],
        on="warehouse_id",
        how="left"
    )
)


warehouse_procurement["on_time_flag"] = (
    warehouse_procurement["supplier_delay_days"] <= 0
).astype(int)

warehouse_procurement["delayed_flag"] = (
    warehouse_procurement["supplier_delay_days"] > 0
).astype(int)


warehouse_procurement_performance = (
    warehouse_procurement
    .groupby(
        [
            "warehouse_id",
            "warehouse_name"
        ]
    )
    .agg(
        po_count=("po_id", "count"),
        ordered_qty=("ordered_qty", "sum"),
        received_qty=("received_qty", "sum"),
        procurement_value=("po_value", "sum"),
        on_time_pos=("on_time_flag", "sum"),
        delayed_pos=("delayed_flag", "sum"),
        avg_delay_days=("supplier_delay_days", "mean")
    )
    .reset_index()
)


warehouse_procurement_performance[
    "outstanding_qty"
] = (
    warehouse_procurement_performance["ordered_qty"]
    - warehouse_procurement_performance["received_qty"]
)


warehouse_procurement_performance[
    "on_time_rate_pct"
] = (
    warehouse_procurement_performance["on_time_pos"]
    / warehouse_procurement_performance["po_count"]
    * 100
)


warehouse_procurement_performance[
    "delay_rate_pct"
] = (
    warehouse_procurement_performance["delayed_pos"]
    / warehouse_procurement_performance["po_count"]
    * 100
)


warehouse_procurement_performance = (
    warehouse_procurement_performance
    .sort_values(
        "on_time_rate_pct",
        ascending=False
    )
    .reset_index(drop=True)
)


print("WAREHOUSE PROCUREMENT PERFORMANCE\n")

print(
    warehouse_procurement_performance.to_string(
        index=False,
        formatters={
            "procurement_value":
                lambda x: f"{x:,.2f}",

            "on_time_rate_pct":
                lambda x: f"{x:.2f}%",

            "delay_rate_pct":
                lambda x: f"{x:.2f}%",

            "avg_delay_days":
                lambda x: f"{x:.2f}"
        }
    )
)


print(
    "\nMissing Warehouse Matches :",
    warehouse_procurement[
        "warehouse_name"
    ].isna().sum()
)

WAREHOUSE PROCUREMENT PERFORMANCE

warehouse_id    warehouse_name  po_count  ordered_qty  received_qty procurement_value  on_time_pos  delayed_pos avg_delay_days  outstanding_qty on_time_rate_pct delay_rate_pct
        WH02       Mumbai West      1382       325677        305481  1,056,161,950.23          922          460           0.90            20196           66.71%         33.29%
        WH06           Pune DC      1375       328192        307832  1,087,038,463.69          917          458           0.77            20360           66.69%         33.31%
        WH03         Delhi NCR      1430       330101        308880  1,080,897,189.29          934          496           0.76            21221           65.31%         34.69%
        WH04   Hyderabad South      1473       353552        331168  1,212,229,344.06          955          518           0.75            22384           64.83%         35.17%
        WH05       Chennai Hub      1389       327257        306975  1,101,891,720.31

## Step 15 — Monthly Procurement Trend Analysis

Analyze procurement activity over time to understand changes in purchasing volume and value.

For each purchase-order month, evaluate:

- Number of purchase orders
- Ordered quantity
- Received quantity
- Procurement value
- Average purchase order value

This provides visibility into procurement demand patterns and purchasing activity over time.

In [15]:
# Monthly Procurement Trend Analysis

purchase_orders["po_year_month"] = (
    purchase_orders["po_date"]
    .dt.to_period("M")
    .astype(str)
)


monthly_procurement = (
    purchase_orders
    .groupby("po_year_month")
    .agg(
        po_count=("po_id", "count"),
        ordered_qty=("ordered_qty", "sum"),
        received_qty=("received_qty", "sum"),
        procurement_value=("po_value", "sum")
    )
    .reset_index()
)


monthly_procurement["avg_po_value"] = (
    monthly_procurement["procurement_value"]
    / monthly_procurement["po_count"]
)


monthly_procurement = (
    monthly_procurement
    .sort_values("po_year_month")
    .reset_index(drop=True)
)


print("MONTHLY PROCUREMENT TREND ANALYSIS\n")

print(
    monthly_procurement.to_string(
        index=False,
        formatters={
            "procurement_value":
                lambda x: f"{x:,.2f}",

            "avg_po_value":
                lambda x: f"{x:,.2f}"
        }
    )
)


print(
    f"\nMonths Analyzed : "
    f"{len(monthly_procurement):,}"
)

MONTHLY PROCUREMENT TREND ANALYSIS

po_year_month  po_count  ordered_qty  received_qty procurement_value avg_po_value
      2024-01       263        63883         59992    217,652,910.91   827,577.61
      2024-02       233        59287         55425    182,685,745.69   784,058.99
      2024-03       265        64070         60128    209,374,130.09   790,091.06
      2024-04       269        63053         59113    210,732,353.44   783,391.65
      2024-05       267        60536         56466    200,485,465.86   750,881.89
      2024-06       270        60261         56701    194,650,281.17   720,926.97
      2024-07       253        59163         55368    193,186,164.30   763,581.68
      2024-08       247        59022         55040    190,283,720.89   770,379.44
      2024-09       222        53329         49981    192,063,304.51   865,150.02
      2024-10       263        62182         58370    202,335,716.27   769,337.32
      2024-11       255        59721         55945    197,611,

## Step 16 — Yearly Procurement Performance

Summarize procurement activity by year to evaluate purchasing volume and value trends.

The analysis includes:

- Purchase order count
- Ordered quantity
- Received quantity
- Outstanding quantity
- Procurement value
- Average purchase order value
- Quantity receipt rate

**Note:** 2026 contains data through September only and is therefore treated as a partial-year (YTD) period.

In [16]:
# Yearly Procurement Performance

purchase_orders["po_year"] = (
    purchase_orders["po_date"].dt.year
)


yearly_procurement = (
    purchase_orders
    .groupby("po_year")
    .agg(
        po_count=("po_id", "count"),
        ordered_qty=("ordered_qty", "sum"),
        received_qty=("received_qty", "sum"),
        procurement_value=("po_value", "sum")
    )
    .reset_index()
)


yearly_procurement["outstanding_qty"] = (
    yearly_procurement["ordered_qty"]
    - yearly_procurement["received_qty"]
)


yearly_procurement["receipt_rate_pct"] = (
    yearly_procurement["received_qty"]
    / yearly_procurement["ordered_qty"]
    * 100
)


yearly_procurement["avg_po_value"] = (
    yearly_procurement["procurement_value"]
    / yearly_procurement["po_count"]
)


print("YEARLY PROCUREMENT PERFORMANCE\n")

print(
    yearly_procurement.to_string(
        index=False,
        formatters={
            "procurement_value":
                lambda x: f"{x:,.2f}",

            "receipt_rate_pct":
                lambda x: f"{x:.2f}%",

            "avg_po_value":
                lambda x: f"{x:,.2f}"
        }
    )
)

print(
    "\nNote: 2026 represents YTD data through September."
)

YEARLY PROCUREMENT PERFORMANCE

 po_year  po_count  ordered_qty  received_qty procurement_value  outstanding_qty receipt_rate_pct avg_po_value
    2024      3108       737041        690737  2,425,524,916.04            46304           93.72%   780,413.42
    2025      3151       750078        703476  2,545,220,586.91            46602           93.79%   807,750.11
    2026      2241       524962        485042  1,744,936,854.84            39920           92.40%   778,642.06

Note: 2026 represents YTD data through September.


## Step 17 — Comparable Jan–Sep Procurement Trend

Compare procurement performance across equivalent January–September periods for 2024, 2025, and 2026.

Using the same nine-month period avoids misleading comparisons between complete historical years and partial 2026 data.

The analysis evaluates:

- Purchase order count
- Ordered quantity
- Received quantity
- Outstanding quantity
- Procurement value
- Quantity receipt rate
- Year-over-year procurement value growth

In [17]:
# Comparable Jan-Sep Procurement Trend

jan_sep_procurement = purchase_orders[
    purchase_orders["po_date"].dt.month <= 9
].copy()


comparable_procurement = (
    jan_sep_procurement
    .groupby("po_year")
    .agg(
        po_count=("po_id", "count"),
        ordered_qty=("ordered_qty", "sum"),
        received_qty=("received_qty", "sum"),
        procurement_value=("po_value", "sum")
    )
    .reset_index()
)


comparable_procurement["outstanding_qty"] = (
    comparable_procurement["ordered_qty"]
    - comparable_procurement["received_qty"]
)


comparable_procurement["receipt_rate_pct"] = (
    comparable_procurement["received_qty"]
    / comparable_procurement["ordered_qty"]
    * 100
)


comparable_procurement["yoy_value_growth_pct"] = (
    comparable_procurement["procurement_value"]
    .pct_change()
    * 100
)


print("COMPARABLE JAN-SEP PROCUREMENT TREND\n")

print(
    comparable_procurement.to_string(
        index=False,
        formatters={
            "procurement_value":
                lambda x: f"{x:,.2f}",

            "receipt_rate_pct":
                lambda x: f"{x:.2f}%",

            "yoy_value_growth_pct":
                lambda x: (
                    "-"
                    if pd.isna(x)
                    else f"{x:.2f}%"
                )
        }
    )
)

COMPARABLE JAN-SEP PROCUREMENT TREND

 po_year  po_count  ordered_qty  received_qty procurement_value  outstanding_qty receipt_rate_pct yoy_value_growth_pct
    2024      2289       542604        508214  1,791,114,076.86            34390           93.66%                  NaN
    2025      2370       564014        529452  1,921,051,967.49            34562           93.87%                7.25%
    2026      2241       524962        485042  1,744,936,854.84            39920           92.40%               -9.17%


## Step 18 — Open Purchase Order Analysis

Analyze currently open purchase orders to quantify unresolved procurement commitments.

The analysis evaluates:

- Number of open purchase orders
- Total open ordered quantity
- Total open procurement value
- Average open PO value
- Number of suppliers involved
- Number of products involved
- Number of warehouses affected
- Largest open purchase orders by value

Open purchase orders do not yet have actual receipt dates or supplier delay values, so they are analyzed separately from completed receipt-performance metrics.

In [18]:
# Open Purchase Order Analysis

open_pos = purchase_orders[
    purchase_orders["po_status"] == "Open"
].copy()


open_po_count = len(open_pos)

open_ordered_qty = (
    open_pos["ordered_qty"].sum()
)

open_procurement_value = (
    open_pos["po_value"].sum()
)

avg_open_po_value = (
    open_procurement_value / open_po_count
    if open_po_count > 0
    else 0
)

open_suppliers = (
    open_pos["supplier_id"].nunique()
)

open_products = (
    open_pos["product_id"].nunique()
)

open_warehouses = (
    open_pos["warehouse_id"].nunique()
)


print("OPEN PURCHASE ORDER ANALYSIS\n")

print(
    f"Open Purchase Orders       : "
    f"{open_po_count:,}"
)

print(
    f"Open Ordered Quantity      : "
    f"{open_ordered_qty:,}"
)

print(
    f"Open Procurement Value     : "
    f"{open_procurement_value:,.2f}"
)

print(
    f"Average Open PO Value      : "
    f"{avg_open_po_value:,.2f}"
)

print(
    f"Suppliers with Open POs    : "
    f"{open_suppliers:,}"
)

print(
    f"Products with Open POs     : "
    f"{open_products:,}"
)

print(
    f"Warehouses with Open POs   : "
    f"{open_warehouses:,}"
)


# Largest open POs by procurement value

top_open_pos = (
    open_pos
    .sort_values(
        "po_value",
        ascending=False
    )
    [
        [
            "po_id",
            "supplier_id",
            "product_id",
            "warehouse_id",
            "po_date",
            "expected_date",
            "ordered_qty",
            "po_value"
        ]
    ]
    .head(10)
)


print("\nTOP 10 OPEN PURCHASE ORDERS BY VALUE\n")

print(
    top_open_pos.to_string(
        index=False,
        formatters={
            "po_value":
                lambda x: f"{x:,.2f}"
        }
    )
)

OPEN PURCHASE ORDER ANALYSIS

Open Purchase Orders       : 30
Open Ordered Quantity      : 6,873
Open Procurement Value     : 21,754,641.44
Average Open PO Value      : 725,154.71
Suppliers with Open POs    : 21
Products with Open POs     : 25
Warehouses with Open POs   : 6

TOP 10 OPEN PURCHASE ORDERS BY VALUE

   po_id supplier_id product_id warehouse_id    po_date expected_date  ordered_qty     po_value
PO007488      SUP045    SKU0442         WH06 2026-09-09    2026-10-03          370 2,198,706.50
PO001608      SUP051    SKU0216         WH02 2026-09-13    2026-10-05          447 1,966,674.84
PO003014      SUP092    SKU0459         WH03 2026-08-31    2026-09-17          320 1,824,307.20
PO007798      SUP001    SKU0268         WH02 2026-09-09    2026-09-30          292 1,815,603.44
PO002944      SUP045    SKU0029         WH05 2026-09-09    2026-10-03          299 1,686,554.35
PO003681      SUP030    SKU0439         WH05 2026-09-14    2026-10-01          355 1,356,305.90
PO002523      

## Step 19 — Supplier Concentration Analysis

Evaluate how procurement value is distributed across suppliers.

The analysis identifies:

- Total procurement value by supplier
- Supplier share of total procurement value
- Cumulative procurement value share
- Top 10 suppliers by procurement value
- Combined procurement share of the Top 5 and Top 10 suppliers

This helps assess supplier concentration and dependency risk.

In [19]:
# Supplier Concentration Analysis

supplier_concentration = (
    purchase_orders
    .groupby("supplier_id")
    .agg(
        po_count=("po_id", "count"),
        procurement_value=("po_value", "sum")
    )
    .reset_index()
)


supplier_concentration = supplier_concentration.merge(
    suppliers[
        [
            "supplier_id",
            "supplier_name"
        ]
    ],
    on="supplier_id",
    how="left"
)


supplier_concentration["procurement_share_pct"] = (
    supplier_concentration["procurement_value"]
    / total_procurement_value
    * 100
)


supplier_concentration = (
    supplier_concentration
    .sort_values(
        "procurement_value",
        ascending=False
    )
    .reset_index(drop=True)
)


supplier_concentration["cumulative_share_pct"] = (
    supplier_concentration[
        "procurement_share_pct"
    ].cumsum()
)


top_5_supplier_share = (
    supplier_concentration
    .head(5)["procurement_share_pct"]
    .sum()
)


top_10_supplier_share = (
    supplier_concentration
    .head(10)["procurement_share_pct"]
    .sum()
)


print("SUPPLIER CONCENTRATION ANALYSIS\n")

print(
    supplier_concentration[
        [
            "supplier_id",
            "supplier_name",
            "po_count",
            "procurement_value",
            "procurement_share_pct",
            "cumulative_share_pct"
        ]
    ]
    .head(10)
    .to_string(
        index=False,
        formatters={
            "procurement_value":
                lambda x: f"{x:,.2f}",

            "procurement_share_pct":
                lambda x: f"{x:.2f}%",

            "cumulative_share_pct":
                lambda x: f"{x:.2f}%"
        }
    )
)


print(
    f"\nTop 5 Supplier Procurement Share  : "
    f"{top_5_supplier_share:.2f}%"
)

print(
    f"Top 10 Supplier Procurement Share : "
    f"{top_10_supplier_share:.2f}%"
)

SUPPLIER CONCENTRATION ANALYSIS

supplier_id supplier_name  po_count procurement_value procurement_share_pct cumulative_share_pct
     SUP089  Supplier 089       257    300,925,064.66                 4.48%                4.48%
     SUP020  Supplier 020       183    170,474,871.78                 2.54%                7.02%
     SUP010  Supplier 010       189    161,598,309.71                 2.41%                9.43%
     SUP045  Supplier 045       132    156,766,514.39                 2.33%               11.76%
     SUP006  Supplier 006       163    149,179,965.35                 2.22%               13.98%
     SUP084  Supplier 084       182    134,599,951.47                 2.00%               15.99%
     SUP077  Supplier 077       125    124,084,111.57                 1.85%               17.83%
     SUP053  Supplier 053       149    121,246,675.17                 1.81%               19.64%
     SUP083  Supplier 083       136    119,743,964.16                 1.78%               21.4

## Step 20 — High-Risk Supplier Identification

Identify suppliers that combine significant procurement exposure with weak delivery performance.

A supplier is classified as high risk when:

- On-time receipt rate is below 60%, OR
- Average supplier delay exceeds 2 days

The analysis evaluates:

- Purchase order volume
- Procurement value
- Procurement value share
- On-time receipt rate
- Delay rate
- Average supplier delay
- Quality rating
- Preferred supplier status
- Reliability factor

This provides a practical supplier-risk shortlist for procurement review and corrective action.

In [20]:
# High-Risk Supplier Identification

supplier_risk = supplier_delivery.copy()


supplier_risk["procurement_share_pct"] = (
    supplier_risk["procurement_value"]
    / supplier_risk["procurement_value"].sum()
    * 100
)


supplier_risk["high_risk_flag"] = np.where(
    (
        (supplier_risk["on_time_rate_pct"] < 60)
        |
        (supplier_risk["avg_delay_days"] > 2)
    ),
    "High Risk",
    "Monitor"
)


high_risk_suppliers = (
    supplier_risk[
        supplier_risk["high_risk_flag"] == "High Risk"
    ]
    .sort_values(
        "procurement_value",
        ascending=False
    )
    .reset_index(drop=True)
)


print("HIGH-RISK SUPPLIER IDENTIFICATION\n")

print(
    f"High-Risk Suppliers : "
    f"{len(high_risk_suppliers):,}"
)


print(
    f"High-Risk Procurement Value : "
    f"{high_risk_suppliers['procurement_value'].sum():,.2f}"
)


print(
    f"High-Risk Procurement Share : "
    f"{high_risk_suppliers['procurement_share_pct'].sum():.2f}%"
)


print("\nTOP HIGH-RISK SUPPLIERS BY PROCUREMENT VALUE\n")


print(
    high_risk_suppliers[
        [
            "supplier_id",
            "supplier_name",
            "po_count",
            "procurement_value",
            "procurement_share_pct",
            "on_time_rate_pct",
            "delay_rate_pct",
            "avg_delay_days",
            "quality_rating",
            "preferred_supplier_flag",
            "reliability_factor"
        ]
    ]
    .head(10)
    .to_string(
        index=False,
        formatters={
            "procurement_value":
                lambda x: f"{x:,.2f}",

            "procurement_share_pct":
                lambda x: f"{x:.2f}%",

            "on_time_rate_pct":
                lambda x: f"{x:.2f}%",

            "delay_rate_pct":
                lambda x: f"{x:.2f}%",

            "avg_delay_days":
                lambda x: f"{x:.2f}"
        }
    )
)

HIGH-RISK SUPPLIER IDENTIFICATION

High-Risk Suppliers : 24
High-Risk Procurement Value : 1,754,502,757.20
High-Risk Procurement Share : 26.21%

TOP HIGH-RISK SUPPLIERS BY PROCUREMENT VALUE

supplier_id supplier_name  po_count procurement_value procurement_share_pct on_time_rate_pct delay_rate_pct avg_delay_days  quality_rating preferred_supplier_flag  reliability_factor
     SUP089  Supplier 089       257    300,925,064.66                 4.50%           40.08%         59.92%           4.05            4.52                       N               0.570
     SUP045  Supplier 045       128    152,263,914.74                 2.27%           54.69%         45.31%           2.08            4.00                       N               0.683
     SUP084  Supplier 084       181    133,954,623.41                 2.00%           56.91%         43.09%           1.31            3.70                       Y               0.857
     SUP073  Supplier 073       148    119,728,381.29                 1.79%  

## Step 21 — Supplier Quality vs Delivery Performance

Evaluate the relationship between supplier quality ratings and delivery performance.

Suppliers are grouped into quality-rating bands:

- Below 3.5
- 3.5–3.99
- 4.0–4.49
- 4.5+

For each quality band, evaluate:

- Number of suppliers
- Purchase orders
- Procurement value
- On-time receipt rate
- Delay rate
- Average supplier delay

This helps determine whether higher supplier quality ratings are also associated with stronger delivery reliability.

In [22]:
# Supplier Quality vs Delivery Performance

supplier_quality_analysis = supplier_delivery.copy()


supplier_quality_analysis["quality_band"] = pd.cut(
    supplier_quality_analysis["quality_rating"],
    bins=[0, 3.5, 4.0, 4.5, np.inf],
    labels=[
        "Below 3.5",
        "3.5-3.99",
        "4.0-4.49",
        "4.5+"
    ],
    right=False
)


quality_performance = (
    supplier_quality_analysis
    .groupby(
        "quality_band",
        observed=True
    )
    .agg(
        supplier_count=("supplier_id", "nunique"),
        po_count=("po_count", "sum"),
        procurement_value=("procurement_value", "sum"),
        on_time_pos=("on_time_pos", "sum"),
        delayed_pos=("delayed_pos", "sum")
    )
    .reset_index()
)


quality_performance["on_time_rate_pct"] = (
    quality_performance["on_time_pos"]
    / quality_performance["po_count"]
    * 100
)


quality_performance["delay_rate_pct"] = (
    quality_performance["delayed_pos"]
    / quality_performance["po_count"]
    * 100
)


# PO-weighted average supplier delay by quality band
quality_delay = (
    supplier_quality_analysis
    .groupby(
        "quality_band",
        observed=True
    )
    .apply(
        lambda x: np.average(
            x["avg_delay_days"],
            weights=x["po_count"]
        )
    )
    .rename("avg_delay_days")
    .reset_index()
)


quality_performance = quality_performance.merge(
    quality_delay,
    on="quality_band",
    how="left"
)


print("SUPPLIER QUALITY VS DELIVERY PERFORMANCE\n")

print(
    quality_performance.to_string(
        index=False,
        formatters={
            "procurement_value":
                lambda x: f"{x:,.2f}",

            "on_time_rate_pct":
                lambda x: f"{x:.2f}%",

            "delay_rate_pct":
                lambda x: f"{x:.2f}%",

            "avg_delay_days":
                lambda x: f"{x:.2f}"
        }
    )
)

SUPPLIER QUALITY VS DELIVERY PERFORMANCE

quality_band  supplier_count  po_count procurement_value  on_time_pos  delayed_pos on_time_rate_pct delay_rate_pct avg_delay_days
   Below 3.5              22      1785  1,342,317,628.37         1199          586           67.17%         32.83%           0.69
    3.5-3.99              24      2048  1,612,738,255.28         1287          761           62.84%         37.16%           1.08
    4.0-4.49              27      2399  2,009,174,241.03         1619          780           67.49%         32.51%           0.58
        4.5+              27      2238  1,729,697,591.67         1430          808           63.90%         36.10%           1.02


## Step 22 — Reliability Factor vs Delivery Performance

Evaluate whether supplier reliability scores are reflected in actual delivery performance.

Suppliers are grouped into reliability bands:

- Below 0.70
- 0.70–0.79
- 0.80–0.89
- 0.90+

For each reliability band, evaluate:

- Number of suppliers
- Purchase orders
- Procurement value
- On-time receipt rate
- Delay rate
- Average supplier delay

This helps determine whether the supplier reliability factor is a meaningful indicator of actual procurement delivery performance.

In [23]:
# Reliability Factor vs Delivery Performance

supplier_reliability_analysis = supplier_delivery.copy()


supplier_reliability_analysis["reliability_band"] = pd.cut(
    supplier_reliability_analysis["reliability_factor"],
    bins=[0, 0.70, 0.80, 0.90, np.inf],
    labels=[
        "Below 0.70",
        "0.70-0.79",
        "0.80-0.89",
        "0.90+"
    ],
    right=False
)


reliability_performance = (
    supplier_reliability_analysis
    .groupby(
        "reliability_band",
        observed=True
    )
    .agg(
        supplier_count=("supplier_id", "nunique"),
        po_count=("po_count", "sum"),
        procurement_value=("procurement_value", "sum"),
        on_time_pos=("on_time_pos", "sum"),
        delayed_pos=("delayed_pos", "sum")
    )
    .reset_index()
)


reliability_performance["on_time_rate_pct"] = (
    reliability_performance["on_time_pos"]
    / reliability_performance["po_count"]
    * 100
)


reliability_performance["delay_rate_pct"] = (
    reliability_performance["delayed_pos"]
    / reliability_performance["po_count"]
    * 100
)


reliability_delay = (
    supplier_reliability_analysis
    .groupby(
        "reliability_band",
        observed=True
    )
    .apply(
        lambda x: np.average(
            x["avg_delay_days"],
            weights=x["po_count"]
        )
    )
    .rename("avg_delay_days")
    .reset_index()
)


reliability_performance = (
    reliability_performance
    .merge(
        reliability_delay,
        on="reliability_band",
        how="left"
    )
)


print("RELIABILITY FACTOR VS DELIVERY PERFORMANCE\n")

print(
    reliability_performance.to_string(
        index=False,
        formatters={
            "procurement_value":
                lambda x: f"{x:,.2f}",

            "on_time_rate_pct":
                lambda x: f"{x:.2f}%",

            "delay_rate_pct":
                lambda x: f"{x:.2f}%",

            "avg_delay_days":
                lambda x: f"{x:.2f}"
        }
    )
)

RELIABILITY FACTOR VS DELIVERY PERFORMANCE

reliability_band  supplier_count  po_count procurement_value  on_time_pos  delayed_pos on_time_rate_pct delay_rate_pct avg_delay_days
      Below 0.70              10       966    955,234,753.04          475          491           49.17%         50.83%           2.92
       0.80-0.89              46      3806  2,790,529,731.22         2418         1388           63.53%         36.47%           1.03
           0.90+              44      3698  2,948,163,232.09         2642         1056           71.44%         28.56%           0.09


## Step 23 — Procurement Risk Summary

Consolidate the major supplier and procurement risk indicators identified throughout the analysis.

The summary focuses on:

- Overall receipt performance
- Supplier delivery reliability
- Severe supplier delays
- Open procurement exposure
- High-risk supplier exposure
- Lowest-performing supplier
- Highest-exposure supplier
- Weakest supplier region
- Weakest product category
- Weakest warehouse
- Preferred supplier performance
- Reliability-factor performance

This provides a concise management view of the major procurement risks requiring attention.

In [24]:
# Procurement Risk Summary

worst_supplier = (
    supplier_delivery
    .sort_values("on_time_rate_pct")
    .iloc[0]
)

largest_supplier = (
    supplier_delivery
    .sort_values(
        "procurement_value",
        ascending=False
    )
    .iloc[0]
)

worst_supplier_region = (
    supplier_region_performance
    .sort_values("on_time_rate_pct")
    .iloc[0]
)

worst_category = (
    category_performance
    .sort_values("on_time_rate_pct")
    .iloc[0]
)

worst_warehouse = (
    warehouse_procurement_performance
    .sort_values("on_time_rate_pct")
    .iloc[0]
)


severe_supplier_delays = (
    delayed_pos["supplier_delay_days"] >= 11
).sum()


print("PROCUREMENT RISK SUMMARY\n")

print(
    f"Overall Quantity Receipt Rate      : "
    f"{quantity_receipt_rate:.2f}%"
)

print(
    f"Supplier On-Time Receipt Rate      : "
    f"{supplier_on_time_rate:.2f}%"
)

print(
    f"Supplier Delay Rate                : "
    f"{supplier_delay_rate:.2f}%"
)

print(
    f"11+ Day Delayed Purchase Orders    : "
    f"{severe_supplier_delays:,}"
)

print(
    f"Open Purchase Orders               : "
    f"{open_po_count:,}"
)

print(
    f"Open Procurement Value             : "
    f"{open_procurement_value:,.2f}"
)

print(
    f"High-Risk Suppliers                : "
    f"{len(high_risk_suppliers):,}"
)

print(
    f"High-Risk Procurement Share        : "
    f"{high_risk_suppliers['procurement_share_pct'].sum():.2f}%"
)


print("\nKEY RISK AREAS\n")


print(
    f"Lowest On-Time Supplier            : "
    f"{worst_supplier['supplier_name']} "
    f"({worst_supplier['on_time_rate_pct']:.2f}%)"
)

print(
    f"Highest Procurement Exposure       : "
    f"{largest_supplier['supplier_name']} "
    f"({largest_supplier['procurement_value']:,.2f})"
)

print(
    f"Weakest Supplier Region            : "
    f"{worst_supplier_region['supplier_region']} "
    f"({worst_supplier_region['on_time_rate_pct']:.2f}%)"
)

print(
    f"Weakest Product Category           : "
    f"{worst_category['category']} "
    f"({worst_category['on_time_rate_pct']:.2f}%)"
)

print(
    f"Weakest Warehouse                  : "
    f"{worst_warehouse['warehouse_name']} "
    f"({worst_warehouse['on_time_rate_pct']:.2f}%)"
)

PROCUREMENT RISK SUMMARY

Overall Quantity Receipt Rate      : 93.40%
Supplier On-Time Receipt Rate      : 65.35%
Supplier Delay Rate                : 34.65%
11+ Day Delayed Purchase Orders    : 514
Open Purchase Orders               : 30
Open Procurement Value             : 21,754,641.44
High-Risk Suppliers                : 24
High-Risk Procurement Share        : 26.21%

KEY RISK AREAS

Lowest On-Time Supplier            : Supplier 012 (38.64%)
Highest Procurement Exposure       : Supplier 089 (300,925,064.66)
Weakest Supplier Region            : West (63.39%)
Weakest Product Category           : Electronics (63.60%)
Weakest Warehouse                  : Bengaluru Central (63.97%)


## Step 24 — Procurement Analysis Reconciliation

Perform reconciliation checks across the major procurement analyses.

The checks confirm that:

- Purchase-order status counts reconcile to the source dataset
- Ordered and received quantities reconcile
- Procurement value reconciles
- Received and open purchase orders reconcile
- On-time and delayed receipts reconcile
- Supplier-level analysis covers all evaluated purchase orders
- Supplier-region analysis reconciles
- Product-category analysis reconciles
- Warehouse analysis reconciles
- Delay-severity classifications reconcile
- Open purchase-order analysis reconciles

These checks ensure analytical consistency before finalizing the notebook.

In [25]:
# Procurement Analysis Reconciliation

reconciliation_checks = {

    "PO Status Count":
        po_status_summary["po_count"].sum()
        == total_purchase_orders,

    "Ordered Quantity":
        po_status_summary["ordered_qty"].sum()
        == total_units_ordered,

    "Received Quantity":
        po_status_summary["received_qty"].sum()
        == total_units_received,

    "Procurement Value":
        np.isclose(
            po_status_summary["procurement_value"].sum(),
            total_procurement_value
        ),

    "Received + Open PO Count":
        evaluated_pos + open_po_count
        == total_purchase_orders,

    "On-Time + Delayed Receipts":
        on_time_supplier_pos + delayed_supplier_pos
        == evaluated_pos,

    "Delay Severity":
        delay_severity_summary["po_count"].sum()
        == delayed_supplier_pos,

    "Supplier Analysis":
        supplier_delivery["po_count"].sum()
        == evaluated_pos,

    "Supplier Region Analysis":
        supplier_region_performance["po_count"].sum()
        == evaluated_pos,

    "Category Analysis":
        category_performance["po_count"].sum()
        == evaluated_pos,

    "Warehouse Analysis":
        warehouse_procurement_performance["po_count"].sum()
        == evaluated_pos,

    "Open PO Count":
        len(open_pos)
        == open_po_count,

    "Open PO Quantity":
        open_pos["ordered_qty"].sum()
        == open_ordered_qty,

    "Open PO Value":
        np.isclose(
            open_pos["po_value"].sum(),
            open_procurement_value
        )
}


print("PROCUREMENT ANALYSIS RECONCILIATION\n")


for check, result in reconciliation_checks.items():

    print(
        f"{check:<35} : "
        f"{'PASS' if result else 'FAIL'}"
    )


procurement_reconciliation_passed = all(
    reconciliation_checks.values()
)


print(
    "\nPROCUREMENT ANALYSIS RECONCILIATION STATUS :",
    "PASSED"
    if procurement_reconciliation_passed
    else "REVIEW REQUIRED"
)

PROCUREMENT ANALYSIS RECONCILIATION

PO Status Count                     : PASS
Ordered Quantity                    : PASS
Received Quantity                   : PASS
Procurement Value                   : PASS
Received + Open PO Count            : PASS
On-Time + Delayed Receipts          : PASS
Delay Severity                      : PASS
Supplier Analysis                   : PASS
Supplier Region Analysis            : PASS
Category Analysis                   : PASS
Warehouse Analysis                  : PASS
Open PO Count                       : PASS
Open PO Quantity                    : PASS
Open PO Value                       : PASS

PROCUREMENT ANALYSIS RECONCILIATION STATUS : PASSED


## Step 25 — Key Business Insights

### 1. Overall Procurement Performance

The organization processed **8,500 purchase orders** with a total procurement value of approximately **6.72 billion**.

A total of **2,012,081 units** were ordered and **1,879,255 units** were received, resulting in an overall **quantity receipt rate of 93.40%**.

Approximately **132,826 units remain outstanding** across the procurement portfolio.

---

### 2. Supplier Delivery Reliability

Among **8,470 purchase orders with completed receipt information**:

- **5,535 POs (65.35%)** were received on time or early.
- **2,935 POs (34.65%)** were delayed.
- Average supplier delay was **0.84 days**.
- Maximum supplier delay reached **17 days**.

Although overall quantity receipt performance is high, supplier delivery reliability remains a significant operational concern.

---

### 3. Severe Supplier Delays

Among delayed purchase orders:

- **1,872 (63.78%)** were delayed by 1–2 days.
- **227 (7.73%)** were delayed by 3–5 days.
- **322 (10.97%)** were delayed by 6–10 days.
- **514 (17.51%)** were delayed by 11+ days.

The **11+ day delay group** represents a particularly important procurement risk because these POs experienced an average delay of **14.21 days**.

---

### 4. Supplier-Level Risk

Supplier performance varies substantially.

**Supplier 012** recorded the lowest on-time receipt rate at **38.64%**.

However, **Supplier 089 represents the most important supplier risk** because it combines:

- Approximately **300.93 million** in procurement exposure
- **257 evaluated purchase orders**
- Only **40.08% on-time performance**
- **59.92% delay rate**
- Average delay of **4.05 days**

Supplier 089 is also the organization's largest supplier by procurement value.

---

### 5. High-Risk Supplier Exposure

Using the defined supplier-risk criteria:

- **24 suppliers** were classified as High Risk.
- These suppliers represent approximately **1.75 billion** in procurement value.
- High-risk suppliers account for approximately **26.21% of procurement exposure**.

This indicates that more than one-quarter of procurement value is associated with suppliers exhibiting weak delivery performance.

---

### 6. Preferred Supplier Performance

Preferred suppliers demonstrate moderately stronger delivery performance than non-preferred suppliers.

- Preferred suppliers: **67.12% on-time**
- Non-preferred suppliers: **64.26% on-time**
- Preferred supplier average delay: **0.57 days**
- Non-preferred supplier average delay: **1.00 day**

Preferred-supplier designation therefore provides some delivery advantage, although the difference is not large enough to eliminate supplier-performance risk.

---

### 7. Supplier Reliability Factor

The supplier reliability factor shows a strong relationship with actual delivery performance.

- Reliability below 0.70 → **49.17% on-time**, average delay **2.92 days**
- Reliability 0.80–0.89 → **63.53% on-time**, average delay **1.03 days**
- Reliability 0.90+ → **71.44% on-time**, average delay only **0.09 days**

This suggests that the reliability factor is a useful indicator for supplier monitoring and procurement decisions.

---

### 8. Supplier Quality Rating

Supplier quality rating does not show the same clear relationship with delivery reliability.

The **4.0–4.49 quality band** achieved the strongest delivery performance at **67.49% on-time**, while suppliers rated **4.5+** achieved only **63.90% on-time**.

Therefore, product/service quality and delivery reliability should be evaluated as separate supplier-performance dimensions.

---

### 9. Regional Supplier Performance

Supplier delivery performance varies moderately across regions.

- **East** achieved the highest on-time rate at **67.35%**.
- **West** recorded the lowest on-time rate at **63.39%** and the highest average delay at **1.09 days**.

The regional differences exist but are smaller than the performance differences observed between individual suppliers.

---

### 10. Product Category Procurement Risk

**Electronics** represents the largest procurement category, accounting for approximately **28.05% of procurement value**.

It also recorded the lowest category-level on-time receipt rate at **63.60%** and the highest average delay at **1.03 days**.

This combination of high procurement exposure and weaker delivery performance makes Electronics an important procurement-risk area.

---

### 11. Warehouse Procurement Performance

Warehouse-level supplier receipt performance is relatively close across the network.

- **Mumbai West** achieved the highest on-time receipt rate at **66.71%**.
- **Bengaluru Central** recorded the lowest rate at **63.97%**.

The relatively narrow warehouse performance range suggests that procurement delays are more strongly associated with supplier performance than with individual warehouse locations.

---

### 12. Procurement Trend

For comparable January–September periods:

- 2025 procurement value increased **7.25%** compared with 2024.
- 2026 procurement value decreased **9.17%** compared with 2025.
- Quantity receipt rate declined from **93.87% in 2025** to **92.40% in 2026**.
- Outstanding quantity increased to **39,920 units** in Jan–Sep 2026.

The decline in receipt rate despite lower procurement volume should be monitored.

---

### 13. Open Purchase Order Exposure

There are currently **30 open purchase orders** representing:

- **6,873 units**
- Approximately **21.75 million** in procurement value
- **21 suppliers**
- **25 products**
- All **6 warehouses**

Open procurement exposure is relatively small compared with the total procurement portfolio but should continue to be monitored until receipt.

---

### 14. Supplier Concentration

Procurement spending is relatively diversified.

- Top 5 suppliers represent **13.98%** of procurement value.
- Top 10 suppliers represent **23.20%**.

Therefore, the organization does not appear excessively dependent on a small supplier group.

---

## Management Takeaway

The procurement operation demonstrates a strong overall **93.40% quantity receipt rate**, but supplier delivery reliability remains the primary concern, with only **65.35% of received POs arriving on time or early**.

The most important improvement opportunities are:

- Reduce exposure to high-risk suppliers.
- Prioritize corrective action for Supplier 089 and other high-value poor-performing suppliers.
- Monitor suppliers with low reliability factors.
- Reduce severe 11+ day supplier delays.
- Strengthen Electronics procurement reliability.
- Review the decline in the 2026 YTD receipt rate.
- Continue monitoring open procurement commitments.

Supplier selection should therefore consider **delivery reliability, procurement exposure, quality, and supplier concentration together**, rather than relying on quality ratings alone.

## Step 26 — Final Supplier & Procurement Analysis Validation

Perform the final validation of the Supplier & Procurement Analysis notebook.

The validation confirms that:

- Purchase-order totals remain unchanged
- Required procurement fields are available
- Structural nulls correspond to open purchase orders
- Procurement quantities and values reconcile
- Received and open purchase orders reconcile
- Supplier delivery classifications reconcile
- Delay-severity analysis reconciles
- Supplier, region, category, and warehouse analyses are complete
- Open purchase-order exposure reconciles
- Supplier-risk classification is valid
- The Step 24 procurement reconciliation passed

This final validation ensures that the notebook is analytically consistent and ready for reporting and dashboard development.

In [26]:
# Final Supplier & Procurement Analysis Validation

final_validation_checks = {

    "Purchase Order Row Count":
        len(purchase_orders) == 8500,

    "Required Columns Available":
        len(missing_columns) == 0,

    "Open PO Count":
        open_po_count == 30,

    "Structural Receipt-Date Nulls":
        purchase_orders[
            "actual_receipt_date"
        ].isna().sum() == open_po_count,

    "Structural Supplier-Delay Nulls":
        purchase_orders[
            "supplier_delay_days"
        ].isna().sum() == open_po_count,

    "Received + Open POs Reconcile":
        evaluated_pos + open_po_count
        == total_purchase_orders,

    "Ordered Quantity Reconciles":
        purchase_orders[
            "ordered_qty"
        ].sum() == total_units_ordered,

    "Received Quantity Reconciles":
        purchase_orders[
            "received_qty"
        ].sum() == total_units_received,

    "Procurement Value Reconciles":
        np.isclose(
            purchase_orders[
                "po_value"
            ].sum(),
            total_procurement_value
        ),

    "Outstanding Quantity Reconciles":
        (
            total_units_ordered
            - total_units_received
        ) == outstanding_units,

    "On-Time + Delayed Reconcile":
        (
            on_time_supplier_pos
            + delayed_supplier_pos
        ) == evaluated_pos,

    "Delay Severity Reconciles":
        (
            delay_severity_summary[
                "po_count"
            ].sum()
            == delayed_supplier_pos
        ),

    "Supplier Analysis Reconciles":
        (
            supplier_delivery[
                "po_count"
            ].sum()
            == evaluated_pos
        ),

    "Supplier Region Reconciles":
        (
            supplier_region_performance[
                "po_count"
            ].sum()
            == evaluated_pos
        ),

    "Category Analysis Reconciles":
        (
            category_performance[
                "po_count"
            ].sum()
            == evaluated_pos
        ),

    "Warehouse Analysis Reconciles":
        (
            warehouse_procurement_performance[
                "po_count"
            ].sum()
            == evaluated_pos
        ),

    "Open PO Quantity Reconciles":
        (
            open_pos[
                "ordered_qty"
            ].sum()
            == open_ordered_qty
        ),

    "Open PO Value Reconciles":
        np.isclose(
            open_pos[
                "po_value"
            ].sum(),
            open_procurement_value
        ),

    "Supplier Count Valid":
        supplier_delivery[
            "supplier_id"
        ].nunique() == 100,

    "High-Risk Supplier Count Valid":
        len(high_risk_suppliers) == 24,

    "Receipt Rate Valid":
        0 <= quantity_receipt_rate <= 100,

    "Supplier On-Time Rate Valid":
        0 <= supplier_on_time_rate <= 100,

    "Step 24 Reconciliation Passed":
        procurement_reconciliation_passed
}


print(
    "FINAL SUPPLIER & PROCUREMENT ANALYSIS VALIDATION\n"
)


for check, result in final_validation_checks.items():

    print(
        f"{check:<42} : "
        f"{'PASS' if result else 'FAIL'}"
    )


final_procurement_status = all(
    final_validation_checks.values()
)


print(
    "\nFINAL SUPPLIER & PROCUREMENT ANALYSIS STATUS :",
    "PASSED"
    if final_procurement_status
    else "REVIEW REQUIRED"
)

FINAL SUPPLIER & PROCUREMENT ANALYSIS VALIDATION

Purchase Order Row Count                   : PASS
Required Columns Available                 : PASS
Open PO Count                              : PASS
Structural Receipt-Date Nulls              : PASS
Structural Supplier-Delay Nulls            : PASS
Received + Open POs Reconcile              : PASS
Ordered Quantity Reconciles                : PASS
Received Quantity Reconciles               : PASS
Procurement Value Reconciles               : PASS
Outstanding Quantity Reconciles            : PASS
On-Time + Delayed Reconcile                : PASS
Delay Severity Reconciles                  : PASS
Supplier Analysis Reconciles               : PASS
Supplier Region Reconciles                 : PASS
Category Analysis Reconciles               : PASS
Warehouse Analysis Reconciles              : PASS
Open PO Quantity Reconciles                : PASS
Open PO Value Reconciles                   : PASS
Supplier Count Valid                       : PASS
